<a href="https://colab.research.google.com/github/arishnaorem08-hash/aimllearn_by_self/blob/main/rice_crop_diseases.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install split-folders

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV3Small
import os
import kagglehub
import splitfolders

# 1. Define Constants
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
CHANNELS = 3
EPOCHS = 10

# Download latest version
path = kagglehub.dataset_download("alamshihab075/rice-leaf-disease-an-images-dataset")
print(f"Dataset downloaded to: {path}")

# Find the correct directory that contains subfolders with images
# Sometimes Kaggle datasets are nested inside a subfolder
input_dir = path
for root, dirs, files in os.walk(path):
    # If we find a directory containing files or nested structures, let's locate class directories
    # Typically, the dataset folder contains subfolders which are class names.
    # Let's check if there are subfolders in 'root' that contain images.
    image_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.gif')
    has_images = any(f.lower().endswith(image_extensions) for f in files)
    if has_images:
        # The parent of the image files is the class folder, and its parent is our target input_dir
        input_dir = os.path.dirname(root)
        break

print(f"Using input directory for splitting: {input_dir}")

# Split the dataset into train and val (80% train, 20% val)
splitfolders.ratio(input_dir, output="dataset", seed=1337, ratio=(.8, .2))

# Update paths to where splitfolders actually placed the class subdirectories
train_dir = 'dataset/train'
val_dir = 'dataset/val'

# Detect number of classes dynamically
NUM_CLASSES = len([d for d in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, d))])
print(f"Detected {NUM_CLASSES} classes.")

# 2. Load and Augment Data
train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_datagen = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

# 3. Build Model using Transfer Learning (MobileNetV3)
base_model = MobileNetV3Small(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

# 4. Compile the Model
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# 5. Train the Model
print("Starting training...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS
)

# 6. Save the Trained Model
model.save('rice_disease_mobilenet.h5')
print("Model successfully saved as rice_disease_mobilenet.h5!")

Using Colab cache for faster access to the 'rice-leaf-disease-an-images-dataset' dataset.
Dataset downloaded to: /kaggle/input/rice-leaf-disease-an-images-dataset
Using input directory for splitting: /kaggle/input/rice-leaf-disease-an-images-dataset/Rice Disease


Copying files: 29363 files [03:03, 160.20 files/s]


Detected 18 classes.
Found 23484 images belonging to 18 classes.
Found 5879 images belonging to 18 classes.
4334752/4334752 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Starting training...
Epoch 1/10
734/734 ━━━━━━━━━━━━━━━━━━━━ 288s 378ms/step - accuracy: 0.1498 - loss: 2.4247 - val_accuracy: 0.1631 - val_loss: 2.3799
Epoch 2/10
734/734 ━━━━━━━━━━━━━━━━━━━━ 267s 364ms/step - accuracy: 0.1696 - loss: 2.3796 - val_accuracy: 0.1771 - val_loss: 2.3565
Epoch 3/10
734/734 ━━━━━━━━━━━━━━━━━━━━ 265s 361ms/step - accuracy: 0.1798 - loss: 2.3477 - val_accuracy: 0.1914 - val_loss: 2.3078
Epoch 4/10
734/734 ━━━━━━━━━━━━━━━━━━━━ 266s 363ms/step - accuracy: 0.1942 - loss: 2.3139 - val_accuracy: 0.2097 - val_loss: 2.2682
Epoch 5/10
697/734 ━━━━━━━━━━━━━━━━━━━━ 11s 317ms/step - accuracy: 0.2022 - loss: 2.2832

### Using the Trained Model for Prediction

You can use the following code to load your saved model (`rice_disease_mobilenet.h5`), load a new query image of a rice leaf, preprocess it to match the expected format (resized to 224x224 and rescaled), and get the predicted disease name.

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os

# Save in the modern native Keras format if we still have the model object in memory
if 'model' in globals():
    print("Re-saving model in native .keras format...")
    model.save('rice_disease_mobilenet.keras')

model_path = 'rice_disease_mobilenet.keras'
if not os.path.exists(model_path) and os.path.exists('rice_disease_mobilenet.h5'):
    # Fallback/convert if only .h5 is available
    print("Loading legacy .h5 model to convert it...")
    try:
        legacy_model = tf.keras.models.load_model('rice_disease_mobilenet.h5')
        legacy_model.save(model_path)
    except Exception as e:
        print("Could not load legacy .h5 directly. If you just finished training, make sure to save the original model object as .keras.")

# 1. Load the saved model in native Keras format
loaded_model = tf.keras.models.load_model(model_path)
print("Model loaded successfully!")

# 2. Get class mapping from your training generator
try:
    class_indices = train_generator.class_indices
    class_names = {v: k for k, v in class_indices.items()}
except NameError:
    class_names = {}

# 3. Helper function to load and predict an image
def predict_crop_disease(image_path):
    # Load and resize image
    img = tf.keras.preprocessing.image.load_img(image_path, target_size=(224, 224))

    # Preprocess the image (rescale by 1/255 and add batch dimension)
    img_array = tf.keras.preprocessing.image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    # Make prediction
    predictions = loaded_model.predict(img_array)
    predicted_class_idx = np.argmax(predictions[0])
    confidence = predictions[0][predicted_class_idx] * 100

    # Get class label name
    predicted_label = class_names.get(predicted_class_idx, f"Class {predicted_class_idx}")

    # Display image and results
    plt.imshow(img)
    plt.title(f"Prediction: {predicted_label} ({confidence:.2f}%)")
    plt.axis('off')
    plt.show()

    return predicted_label, confidence

In [ ]:
import glob

# Find all images in the validation directory
val_images = glob.glob('dataset/val/**/*.jpg', recursive=True) + glob.glob('dataset/val/**/*.png', recursive=True)

if val_images:
    # Select a sample image from the validation set
    sample_image_path = val_images[650]
    print(f"Selected sample image for prediction: {sample_image_path}")

    # Run prediction using the defined helper function
    predict_crop_disease(sample_image_path)
else:
    print("No images found in 'dataset/val'. Please make sure the training and splitting cell (Q4rsUnCrBJ-8) has finished executing.")

### 7. Fine-Tuning the Model
Now, we will unfreeze the base MobileNetV3 layers so that we can train them on our rice leaf disease images.

* **Unfreeze the base model:** We set `base_model.trainable = True`.
* **Lower the learning rate:** It is critical to use a much smaller learning rate (e.g., `1e-5` instead of the default `1e-3`) during fine-tuning so we do not distort the pre-trained ImageNet features too rapidly.
* **Recompile and Train:** We recompile the model and train for a few additional epochs.

In [ ]:
# 1. Unfreeze the base model
base_model.trainable = True

# Let's see how many layers are in the base model
print(f"Number of layers in the base model: {len(base_model.layers)}")

# Optional: Fine-tune only from a certain layer onwards
# We will unfreeze the entire base model but use a very low learning rate
# 2. Recompile with a very low learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Show the updated summary
model.summary()

In [ ]:
# 3. Fine-tune the model for 5 more epochs
FINE_TUNE_EPOCHS = 5
total_epochs = EPOCHS + FINE_TUNE_EPOCHS

print("Starting fine-tuning...")
history_fine = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=total_epochs,
    initial_epoch=history.epoch[-1] + 1 if 'history' in globals() else 10
)

In [ ]:
# 4. Save the fine-tuned model
model.save('rice_disease_mobilenet_finetuned.keras')
print("Fine-tuned model saved successfully as rice_disease_mobilenet_finetuned.keras!")

# Update our active loaded model for the prediction helper
loaded_model = model

In [ ]:
# 5. Test prediction again on the same validation image to check improvements
predict_crop_disease(sample_image_path)

In [ ]:
import tensorflow as tf
from tensorflow.keras import models, layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.applications import MobileNetV3Large

# 1. Re-build with a stronger backbone (MobileNetV3Large)
base_model_large = MobileNetV3Large(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base_model_large.trainable = False

# Build improved classifier head
improved_model = models.Sequential([
    base_model_large,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

# 2. Compile with an optimal initial learning rate
improved_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# 3. Callbacks for higher accuracy and training stability
callbacks = [
    # Automatically stop training when validation loss stops improving
    EarlyStopping(
        monitor='val_loss',
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),
    # Dynamically lower learning rate if performance plateaus
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

# 4. Train the improved model
print("Starting training on improved architecture...")
history_improved = improved_model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=15,
    callbacks=callbacks
)

# 5. Save the improved model
improved_model.save('rice_disease_improved.keras')
print("Improved model saved successfully as rice_disease_improved.keras!")